## 5.3. Formatting Meteorological Datasets for UMEP Processing

First check downloaded .csv is in the right format

In [6]:
import pandas as pd
from datetime import datetime
import os

# ---- CONFIGURATION ----
INPUT_CSV = r"C:\Users\Hojung Yu\Desktop\vip_met\savannah_file6bb62ac1211_lat=32_lng=-81_period=20240601-20240917.csv" # Your CSV file path

df = pd.read_csv(INPUT_CSV, skiprows=11)
df.head()

,datetime_lst,datetime,t2m,stl3,stl4,sp,ssrd,strd,tcc,tp,...,sd,wdir10,ws10,ws100,r2m,alpha_sol,azimuth_sol,CF100,dhi,dni
0,2024-06-01 00:00:00,2024-06-01 05:00:00,22.4,24.3,22.6,102223,0,348,0.98,0.0,...,0,89,5.0,7.0,0.64,0.0,-2.93,0.22,0,0
1,2024-06-01 01:00:00,2024-06-01 06:00:00,21.7,24.3,22.6,102201,0,345,0.66,0.0,...,0,89,4.8,6.8,0.63,0.0,3.11,0.20,0,0
2,2024-06-01 02:00:00,2024-06-01 07:00:00,21.4,24.3,22.6,102164,0,335,0.74,0.0,...,0,89,5.0,7.2,0.62,0.0,2.87,0.25,0,0
3,2024-06-01 03:00:00,2024-06-01 08:00:00,20.8,24.3,22.6,102160,0,332,0.74,0.0,...,0,89,5.1,7.3,0.64,0.0,2.63,0.26,0,0
4,2024-06-01 04:00:00,2024-06-01 09:00:00,20.7,24.3,22.6,102167,0,330,0.90,0.0,...,0,91,4.9,7.2,0.64,0.0,2.40,0.25,0,0


Now, you can format date and time into the right format.

# Selecting times from old Framework

In [21]:
import pandas as pd
from datetime import datetime
import os

# ---- CONFIGURATION ------------------------ #

INPUT_CSV = r"C:\Users\HojungYu\OneDrive\GT_Research_Basu\weather\Boston\historic_20240831.csv"
OUTPUT_TXT = r"C:\Users\HojungYu\OneDrive\GT_Research_Basu\weather\Boston\old_framework_typical.txt"  # Output txt path
DATETIME_FORMAT = "%Y-%m-%d %H:%M:%S"     # format used in input csv datetime column
TIME_ZONE = 'America/New_York'

# This should be Any Local Time (Including Daylight Time)
selected_times_EDT = [
    "2023-08-24 08:00:00",
    "2023-08-24 09:00:00",
    "2023-08-24 10:00:00",
    "2023-08-24 11:00:00",
    "2023-08-24 12:00:00",
    "2023-08-24 13:00:00",
    "2023-08-24 14:00:00",
    "2023-08-24 15:00:00",
    "2023-08-24 16:00:00",
    "2023-08-24 17:00:00",
    "2023-08-24 18:00:00",
]

# -------------------------------------------- #


# Convert to UTC
times = pd.to_datetime(selected_times_EDT)
times_edt = times.tz_localize(TIME_ZONE)
times_utc = times_edt.tz_convert('UTC')

# Get UTC offset from the localized times (handles DST automatically)
utc_offset = times_edt[0].utcoffset()
hours_offset = int(utc_offset.total_seconds() / 3600)
print(f"UTC offset for selected times: UTC{hours_offset:+d}")

# column mappings
valid_columns = {
    'datetime_utc': 'datetime',   # Local DATETIME
    # 'datetime_lst': 'datetime_lst',
    'Tair': 't2m',                 # 2m temperature
    'pres': 'sp',                 # surface pressure
    'kdown': 'ssrd',              # surface solar radiation downwards
    'Idown': 'strd',              # surface long radiation downwards
    'fcld': 'tcc',                 # cloud cover
    'rain': 'tp',                 # total precipitation
    'wdir': 'wdir10',               # wind direction
    'wspeed': 'ws10',                  # 10 m wind speed
    'RH': 'r2m',                   # 2 m relative humidity
    'Kdiff' : 'dhi',
    'Kdir' : 'dni',
    
}

df = pd.read_csv(INPUT_CSV, skiprows=11)
df['datetime'] = pd.to_datetime(df['datetime'], utc=True)

# Filter and compute local time by converting UTC -> TIME_ZONE
df_filtered = df[df[valid_columns['datetime_utc']].isin(times_utc)].copy()
df_filtered['datetime_lst'] = df_filtered['datetime'].dt.tz_convert(TIME_ZONE)

print(f"Selected UTC Time:\n {df_filtered['datetime']}\nLocal Time:\n {df_filtered['datetime_lst']}")

# extract_time_parts now accepts a datetime object directly (not a string)
def extract_time_parts(dt):
    return dt.year, dt.timetuple().tm_yday, dt.hour, dt.minute

df_format = df_filtered.copy()
df_format[['iy', 'id', 'it', 'imin']] = df_filtered['datetime_lst'].apply(
    lambda x: pd.Series(extract_time_parts(x))
)
df_format['pres_kPa'] = df_format[valid_columns['pres']] / 1000.0 # pa -> kpa
df_format['r2m_pct'] = df_format[valid_columns['RH']] * 100.0 # humidity -> %
print(df_format)

OUTPUT_TXT = OUTPUT_TXT[:-4] + f"_UTC{hours_offset:+d}" + OUTPUT_TXT[-4:]
print("Output will be saved to ->", OUTPUT_TXT)

# write to output file
with open(OUTPUT_TXT, 'w') as f:
    # header row
    f.write("%iy  id  it imin   Q*      QH      QE      Qs      Qf    Wind    RH     Td     press   rain    Kdn    snow    ldown   fcld    wuh     xsmd    lai_hr  Kdiff   Kdir    Wd\n")
    
    for _, row in df_format.iterrows():
        line = [
            int(row['iy']), int(row['id']), int(row['it']), int(row['imin']),
            *[-999.00]*5,                          # Q*, QH, QE, Qs, Qf
            float(row[valid_columns['wspeed']]),        # Wind
            float(row['r2m_pct']),       # RH
            float(row[valid_columns['Tair']]),     # Td (temp)
            float(row['pres_kPa']),                # press
            float(row[valid_columns['rain']]),     # rain
            float(row[valid_columns['kdown']]),    # Kdn (Incoming shortwavie radiation)
            *[-999.00]*1,                           # snow up )snow_
            float(row[valid_columns['Idown']]),     #Idown (thermal radiation)
            float(row[valid_columns['fcld']]),      # fcld (cloud fraction)
            *[-999.0]*3,                            # wuh(external water use), xsmd(soil moisture), lai_hr(leaf area imndex)
            float(row[valid_columns['Kdiff']]),    # Kdiff (Diffuse shortawave radation [W m-2])
            float(row[valid_columns['Kdir']]),     # Kdir (Direct shortwave radiation [W m-2])
            float(row[valid_columns['wdir']]),  #Wind direction (°)
        ]
        f.write(" ".join(f"{x:.2f}" if isinstance(x, float) else str(x) for x in line) + "\n")

print("\n Finished")

UTC offset for selected times: UTC-4
Selected UTC Time:
 382591   2023-08-24 12:00:00+00:00
382592   2023-08-24 13:00:00+00:00
382593   2023-08-24 14:00:00+00:00
382594   2023-08-24 15:00:00+00:00
382595   2023-08-24 16:00:00+00:00
382596   2023-08-24 17:00:00+00:00
382597   2023-08-24 18:00:00+00:00
382598   2023-08-24 19:00:00+00:00
382599   2023-08-24 20:00:00+00:00
382600   2023-08-24 21:00:00+00:00
382601   2023-08-24 22:00:00+00:00
Name: datetime, dtype: datetime64[ns, UTC]
Local Time:
 382591   2023-08-24 08:00:00-04:00
382592   2023-08-24 09:00:00-04:00
382593   2023-08-24 10:00:00-04:00
382594   2023-08-24 11:00:00-04:00
382595   2023-08-24 12:00:00-04:00
382596   2023-08-24 13:00:00-04:00
382597   2023-08-24 14:00:00-04:00
382598   2023-08-24 15:00:00-04:00
382599   2023-08-24 16:00:00-04:00
382600   2023-08-24 17:00:00-04:00
382601   2023-08-24 18:00:00-04:00
Name: datetime_lst, dtype: datetime64[ns, America/New_York]
                    datetime_lst                  datetim

# Converting Format for New Framework

Converting the new framework is written in R!!!!!!
The input file should have synthesized data and this function only converts .csv -> .txt format!

In [18]:
import pandas as pd
from datetime import datetime
import os

# ---- CONFIGURATION ----
CITY = "Boston"
YEAR = 2023
TIME_ZONE = 'America/New_York'
INPUT_CSV  = f"C:/Users/HojungYu/OneDrive/GT_Research_Basu/weather/{CITY}/{CITY}_hot_median_20240831.csv"
OUTPUT_TXT = f"C:/Users/HojungYu/OneDrive/GT_Research_Basu/weather/{CITY}/{CITY}_new_hot_median_{YEAR}_formatted.txt"
SKIPROWS = 0
TIME_LIST = [8,9,10,11,12,13,14,15,16,17,18]

valid_columns = {
    'datetime_utc': 'datetime',
    'datetime_lst': 'datetime_lst',
    'Tair': 't2m',
    'pres': 'sp',
    'kdown': 'ssrd',
    'Idown': 'strd',
    'fcld': 'tcc',
    'rain': 'tp',
    'wdir': 'wdir10',
    'wspeed': 'ws10',
    'RH': 'r2m',
    'Kdiff': 'dhi',
    'Kdir': 'dni',
}
# ----------------------

header = ("%iy  id  it imin   Q*      QH      QE      Qs      Qf    Wind    RH     Td     press   rain    "
          "Kdn    snow    ldown   fcld    wuh     xsmd    lai_hr  Kdiff   Kdir    Wd\n")

# Load CSV
df = pd.read_csv(INPUT_CSV, skiprows=SKIPROWS)
df['datetime'] = pd.to_datetime(df[valid_columns['datetime_utc']], utc=True)

# Compute UTC offset from TIME_ZONE and YEAR (handles DST automatically)
sample_time = pd.Timestamp(f"{YEAR}-08-01 12:00:00").tz_localize(TIME_ZONE)
utc_offset = int(sample_time.utcoffset().total_seconds() / 3600)
print(f"UTC offset: UTC{utc_offset:+d}")

# Build times_utc from YEAR + TIME_LIST
selected_times_local = [f"{YEAR}-{m:02d}-{d:02d} {h:02d}:00:00"
                        for m in range(5, 10)
                        for d in range(1, 32)
                        for h in TIME_LIST]
times_local = pd.to_datetime(selected_times_local, errors='coerce').dropna()
times_local = times_local.tz_localize(TIME_ZONE, nonexistent='shift_forward', ambiguous='NaT')
times_utc = times_local.tz_convert('UTC')

# Filter and convert to local time
df_filtered = df[df['datetime'].isin(times_utc)].copy()
df_filtered['datetime_lst'] = df_filtered['datetime'].dt.tz_convert(TIME_ZONE)

# Extract time parts
def time_parts_safe(dt):
    try:
        return pd.Series([dt.year, dt.timetuple().tm_yday, dt.hour, dt.minute],
                         index=['iy', 'id', 'it', 'imin'])
    except Exception:
        return pd.Series([-999, -999, -999, -999], index=['iy', 'id', 'it', 'imin'])

df_format = df_filtered.copy()
df_format[['iy', 'id', 'it', 'imin']] = df_filtered['datetime_lst'].apply(time_parts_safe)
df_format['pres_kPa'] = df_format[valid_columns['pres']] / 1000.0
df_format['r2m_pct']  = df_format[valid_columns['RH']] * 100.0
df_format = df_format[(df_format['iy'] == YEAR) & (df_format['it'].isin(TIME_LIST))]

# Write output
rows_written = 0
# Update output filename with UTC offset
OUTPUT_TXT = OUTPUT_TXT[:-4] + f"_UTC{utc_offset:+d}.txt"
with open(OUTPUT_TXT, 'w', newline='\n') as f:
    f.write(header)
    for _, row in df_format.iterrows():
        try:
            line = [
                int(row['iy']), int(row['id']), int(row['it']), int(row['imin']),
                *[-999.00]*5,
                float(row[valid_columns['wspeed']]),
                float(row['r2m_pct']),
                float(row[valid_columns['Tair']]),
                float(row['pres_kPa']),
                float(row[valid_columns['rain']]),
                float(row[valid_columns['kdown']]),
                *[-999.00],
                float(row[valid_columns['Idown']]),
                float(row[valid_columns['fcld']]),
                *[-999.0]*3,
                float(row[valid_columns['Kdiff']]),
                float(row[valid_columns['Kdir']]),
                float(row[valid_columns['wdir']]),
            ]
            f.write(" ".join(f"{x:.2f}" if isinstance(x, float) else str(x) for x in line) + "\n")
            rows_written += 1
        except Exception:
            continue


print(f"[OK] {os.path.basename(INPUT_CSV)} -> {os.path.basename(OUTPUT_TXT)} ({rows_written} rows)")

UTC offset: UTC-4
[OK] Boston_hot_median_20240831.csv -> Boston_new_hot_median_2023_formatted_UTC-4.txt (11 rows)
